# 7차시 실습 — 내 MVP를 '에이전트'로 (ReAct 루프)

> 1일차에서 만든 추천 앱(예: 점심 추천기)·팀 MVP를, 스스로 도구를 쓰는 **에이전트**로 키웁니다.
> 오늘은 공통 예시 **"맛집 추천 도우미"**로 ReAct 루프를 익히고, 마지막에 **내 팀 MVP에 그대로 적용**합니다.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 실행하세요.


In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

- 1일차: 버튼 누르면 메뉴를 **랜덤 추천**하는 앱을 만들었습니다(정해진 동작).
- 2일차: 같은 앱을 **에이전트**로 — 사용자의 자유로운 요청("예산 2만원, 강남, 매운 거")을 받아 **스스로 도구를 골라** 답하게 만듭니다.

> 공통 예시 = **맛집 추천 도우미**. 구조는 동일하니 **당신 팀 MVP의 도구**로 바꾸면 그대로 적용됩니다.


## STEP 1 — 도구(tool) 정의 — *내 앱이 할 수 있는 일*

에이전트의 **손발**입니다. 1일차에서 만든 앱의 기능을 '도구'로 노출한다고 생각하세요.

⌨️ **터미널 Codex:** `codex "맛집 검색·메뉴조회·예산계산 도구를 가진 추천 에이전트를 만들어줘"`


In [2]:
# 데모용 가짜 데이터 (실제로는 day1 앱의 DB/API 자리)
RESTAURANTS = {
  "강남": [{"name":"매운갈비찜집","price":18000,"taste":"매움"},
          {"name":"순한국밥","price":9000,"taste":"순함"},
          {"name":"불막창","price":22000,"taste":"매움"}],
  "홍대": [{"name":"치즈파스타","price":15000,"taste":"순함"}],
}
def search_restaurants(area: str, taste: str = "") -> str:
    items = RESTAURANTS.get(area, [])
    if taste: items = [r for r in items if r["taste"]==taste]
    return json.dumps(items, ensure_ascii=False) if items else "결과 없음"
def budget_check(price: int, people: int) -> str:
    return f"1인당 {price//max(people,1):,}원"

TOOL_IMPL = {"search_restaurants": search_restaurants, "budget_check": budget_check}
TOOLS = [
 {"type":"function","function":{"name":"search_restaurants","description":"지역(과 맛)으로 맛집 검색",
   "parameters":{"type":"object","properties":{"area":{"type":"string"},"taste":{"type":"string","description":"매움/순함, 선택"}},"required":["area"]}}},
 {"type":"function","function":{"name":"budget_check","description":"총가격과 인원으로 1인당 가격 계산",
   "parameters":{"type":"object","properties":{"price":{"type":"integer"},"people":{"type":"integer"}},"required":["price","people"]}}},
]
print("도구 준비:", list(TOOL_IMPL))

도구 준비: ['search_restaurants', 'budget_check']


## STEP 2 — ReAct 루프: 추론(Reason) → 도구 → 관찰 → 반복

모델이 스스로 도구를 호출하고, 결과를 다시 받아 더 이상 호출이 없을 때까지 반복합니다.


In [3]:
def run_agent(question, max_steps=5, verbose=True):
    messages=[{"role":"system","content":"너는 맛집 추천 도우미다. 필요하면 도구를 쓰고, 충분하면 한국어로 추천 이유와 함께 답하라."},
              {"role":"user","content":question}]
    for step in range(1, max_steps+1):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            if verbose: print(f"\n✅ [최종 답]\n{msg.content}")
            return msg.content
        for tc in msg.tool_calls:
            args=json.loads(tc.function.arguments); result=TOOL_IMPL[tc.function.name](**args)
            if verbose: print(f"🧠 step{step} 도구: {tc.function.name}({args}) → 👀 {result}")
            messages.append({"role":"tool","tool_call_id":tc.id,"content":str(result)})
    return "최대 스텝 도달"

run_agent("강남에서 매운 점심 추천해줘. 2명이서 갈 건데 1인당 얼마야?")

🧠 step1 도구: search_restaurants({'area': '강남', 'taste': '매움'}) → 👀 [{"name": "매운갈비찜집", "price": 18000, "taste": "매움"}, {"name": "불막창", "price": 22000, "taste": "매움"}]
🧠 step2 도구: budget_check({'price': 18000, 'people': 2}) → 👀 1인당 9,000원

✅ [최종 답]
강남에서 매운 점심으로 추천하는 맛집은 **매운갈비찜집**입니다. 

- **가격**: 1인당 9,000원
- **음식**: 매운 갈비찜으로, 술안주와 함께 점심 식사로도 훌륭합니다.

또 다른 선택으로는 **불막창**이 있는데, 가격은 조금 더 비쌉니다(22,000원), 그래서 1인당 11,000원이 됩니다.

매운 음식을 좋아한다면 두 곳 모두 좋은 선택이니, 원하시는 스타일에 맞게 방문해 보세요!


'강남에서 매운 점심으로 추천하는 맛집은 **매운갈비찜집**입니다. \n\n- **가격**: 1인당 9,000원\n- **음식**: 매운 갈비찜으로, 술안주와 함께 점심 식사로도 훌륭합니다.\n\n또 다른 선택으로는 **불막창**이 있는데, 가격은 조금 더 비쌉니다(22,000원), 그래서 1인당 11,000원이 됩니다.\n\n매운 음식을 좋아한다면 두 곳 모두 좋은 선택이니, 원하시는 스타일에 맞게 방문해 보세요!'

## STEP 3 — iterate: 요청을 바꿔보기

 "작게 요청 → 확인 → 수정" 리듬 그대로. 같은 에이전트에 다른 요청을 줘보세요.


In [4]:
run_agent("홍대에서 순한 음식, 1만5천원 예산이면 괜찮아? 3명이야.")

🧠 step1 도구: search_restaurants({'area': '홍대', 'taste': '순함'}) → 👀 [{"name": "치즈파스타", "price": 15000, "taste": "순함"}]
🧠 step1 도구: budget_check({'price': 15000, 'people': 3}) → 👀 1인당 5,000원

✅ [최종 답]
홍대에서 순한 음식을 찾으셨군요! "치즈파스타"가 추천됩니다. 가격은 15,000원으로, 부드러운 맛이 특징입니다.

예산이 1인당 5,000원이지만, 3명이라면 15,000원의 비용으로 충분합니다. 치즈파스타와 같은 메뉴로 적당한 가격과 맛을 즐기실 수 있을 것입니다. 즐거운 식사 되세요!


'홍대에서 순한 음식을 찾으셨군요! "치즈파스타"가 추천됩니다. 가격은 15,000원으로, 부드러운 맛이 특징입니다.\n\n예산이 1인당 5,000원이지만, 3명이라면 15,000원의 비용으로 충분합니다. 치즈파스타와 같은 메뉴로 적당한 가격과 맛을 즐기실 수 있을 것입니다. 즐거운 식사 되세요!'

## STEP 4 — 즉답 vs 단계적 사고 (Reasoning 체감)

강의의 플라밍고 문제를 직접 던져봅니다. **같은 모델**이라도 '단계를 펼치게' 하면 답이 달라집니다.
(토큰을 더 쓰게 한다 = test-time compute)

In [ ]:
Q = "플라밍고 3마리가 있었다. 2마리가 날아가고, 남은 1마리는 바위 뒤에 숨었다. 지금 눈에 보이는 플라밍고는 몇 마리인가?"

def ask(system):
    r = client.chat.completions.create(model=MODEL, temperature=0,
        messages=[{"role":"system","content":system},{"role":"user","content":Q}])
    return r.choices[0].message.content

print("[즉답 강제]", ask("생각 과정 없이, 숫자 답만 한 단어로 말하라."))
print()
print("[단계적 사고]", ask("먼저 단계별로 생각을 적고, 마지막 줄에 '답: N마리'로 답하라."))
# 관찰: 즉답은 패턴('3마리')으로 틀리기 쉽고, 단계를 펼치면 0마리에 도달한다

## STEP 5 — 루프 고장내보기 (종료 조건 & max_steps)

강의에서 본 실패 ①(무한 루프 방지)을 체감합니다. `max_steps=1`로 줄이면 도구를 다 못 쓰고 루프가 끊깁니다.

In [ ]:
# 같은 질문을 스텝 상한만 바꿔 비교 — 로그에서 어디서 끊기는지 관찰
print("=== max_steps=1 (도구를 다 못 씀) ===")
run_agent("강남에서 매운 거 먹고 싶은데 2만원 아래로만.", max_steps=1)
print()
print("=== max_steps=5 (정상) ===")
run_agent("강남에서 매운 거 먹고 싶은데 2만원 아래로만.", max_steps=5)
# 관찰: 상한은 '무한 루프 보험'이지만 너무 낮으면 일을 못 끝낸다 — 작업 복잡도에 맞게

## STEP 6 — ⭐ 내 팀 MVP에 적용

위 구조(도구 정의 → ReAct 루프)는 **그대로** 팀 MVP에 옮길 수 있습니다.
아래 `MY_TOOLS_TODO`를 팀 서비스에 맞게 채워보세요. (예: 일정앱이면 `add_event`, 가계부면 `add_expense`)

⌨️ **터미널 Codex:** `codex "우리 MVP의 핵심 기능 2개를 function tool로 정의하고 ReAct 에이전트에 연결해줘"`


In [ ]:
# 팀별 작성: 우리 MVP가 가진 '도구' 2개를 적어보세요 (이름/설명/인자)
MY_TOOLS_TODO = [
  # {"name": "...", "desc": "...", "args": {"...": "..."}},
  # {"name": "...", "desc": "...", "args": {"...": "..."}},
]
print("우리 MVP 도구 후보:", MY_TOOLS_TODO or "⬜ 채워주세요")
print("→ 채운 뒤 STEP 1의 TOOLS/TOOL_IMPL 형식으로 바꿔 run_agent를 다시 돌리면 '내 MVP 에이전트' 완성")